# 06 — Sélection de la meilleure configuration de prétraitement (sur val)

```text
Sélection de la meilleure configuration de prétraitement sur le split VALIDATION (énoncé §4.4.4).

Chaque candidat = (encodeur, mode de retrieval) :
  img2img : image requête -> image KB la plus proche -> sa 1re caption
  img2txt : image requête -> caption KB la plus proche dans l'espace texte de l'encodeur (CLIP/SigLIP seulement)
Critère principal : CIDEr moyen (par image) sur val ; départage : Jaccard des classes COCO @1.
Protocole de l'énoncé : moyenne sur 10 plis du split de validation + IC95 % bootstrap + test apparié vs le meilleur.
Seules les références de VAL sont utilisées ; le test n'est jamais touché ici.
```

> Projet : étude comparative de techniques RAG pour le captioning d'images (MS COCO) — **groupe G1**. Exécuter les notebooks dans l'ordre : 00 → 01 → 01b → 02 → … → 09. Les helpers partagés sont dans `common.py`, `visual.py`, `rag_data.py`.

In [8]:
import os
# --- paramètre : "config.yaml" = vraies données COCO ; "config_test.yaml" = mini-COCO synthétique (test sans réseau)
CONFIG = os.environ.get("RAG_CONFIG", "config.yaml")
os.environ["RAG_CONFIG"] = CONFIG
print("config utilisée :", CONFIG)

config utilisée : config.yaml


## Imports et constantes

In [9]:
import itertools
import numpy as np
from scipy import stats as st
from common import clean_caption, get_logger, load_config, paths, read_json, set_seed, write_json
from rag_data import Release
log = get_logger("select")

### `per_image_cider()`

In [10]:
def per_image_cider(res, refs):
    from pycocoevalcap.cider.cider import Cider
    ids = sorted(refs)
    _, per = Cider().compute_score({i: [clean_caption(x) for x in refs[i]] for i in ids}, {i: [clean_caption(res[i])] for i in ids})
    return np.array(per)

### `main()`

In [11]:
def main():
    cfg = load_config(); p = paths(cfg); set_seed(cfg["seed"])
    qid = None; rows = []; per = {}
    for enc, ecfg in cfg["encoders"].items():
        if not (p["emb"] / enc / "kb_img.npy").exists():
            continue
        R = Release(p["art"], enc); qid = R.ids("val"); Q = R.image_embeddings("val"); refs = R.eval_refs("val"); M = R.metadata()
        modes = ["img2img"] + (["img2txt"] if (p["emb"] / enc / "kb_txt.npy").exists() else [])
        for mode in modes:
            if mode == "img2img":
                _, I = R.search_images(Q, k=1, query_ids=qid); nn_img = I[:, 0]
                # Robustesse : si le plus proche voisin n'a aucune légende dans la KB (image présente dans l'index mais sans légende),
                # on prend le premier voisin suivant qui en a une, et on le journalise pour l'audit.
                _, I = R.search_images(Q, k=5, query_ids=qid)
                nn_img = np.zeros(len(qid), dtype=int); res = {}; skipped = []
                for n, i in enumerate(qid):
                    for rank, x in enumerate(int(v) for v in I[n]):
                        caps = R.kb_captions_of(x) if x >= 0 else []
                        if caps:
                            nn_img[n] = x; res[int(i)] = caps[0]
                            if rank > 0:
                                skipped.append((int(i), int(I[n][0]), x))
                            break
                    else:
                        raise RuntimeError(f"aucun des 5 voisins de {int(i)} n'a de légende : index et légendes incohérents")
                if skipped:
                    log.warning("img2img [%s] : %d/%d requêtes dont le 1er voisin n'a pas de légende (id_requête, voisin ignoré, voisin retenu) : %s",
                                enc, len(skipped), len(qid), skipped[:10])
            else:
                _, cids, iids = R.search_texts(Q, k=1); nn_img = iids[:, 0]
                capmap = {c["caption_id"]: c["caption_raw"] for c in R.kb_captions()}
                res = {int(i): capmap[int(cids[n, 0])] for n, i in enumerate(qid)}
            sc = per_image_cider(res, {int(i): refs[int(i)] for i in qid})
            jac = np.array([len(set(M[int(q)]["category_counts"]) & set(M[int(j)]["category_counts"])) /
                            max(1, len(set(M[int(q)]["category_counts"]) | set(M[int(j)]["category_counts"]))) for q, j in zip(qid, nn_img)])
            folds = np.array_split(np.random.RandomState(cfg["seed"]).permutation(len(sc)), 10)
            fold_means = [sc[f].mean() for f in folds]
            bs = [sc[np.random.RandomState(b).randint(0, len(sc), len(sc))].mean() for b in range(2000)]
            key = f"{enc}|{mode}"; per[key] = sc
            rows.append({"config": key, "encoder": enc, "mode": mode, "cider_mean": float(sc.mean()), "cider_cv10_mean": float(np.mean(fold_means)),
                         "cider_cv10_std": float(np.std(fold_means)), "ci95": [float(np.percentile(bs, 2.5)), float(np.percentile(bs, 97.5))],
                         "jaccard_top1": float(jac.mean())})
    rows.sort(key=lambda r: (-round(r["cider_mean"], 4), -r["jaccard_top1"]))
    best = rows[0]["config"]
    for r in rows[1:]:
        a, b = per[best], per[r["config"]]
        d = a - b
        r["vs_best"] = {"mean_diff": float(d.mean()), "wilcoxon_p": float(st.wilcoxon(d).pvalue) if np.any(d != 0) else 1.0}
    prim = cfg["primary_encoder"]; rec = rows[0]
    if rows[0]["encoder"] != prim:                    # le meilleur n'est pas l'encodeur principal : écart significatif ?
        close = [r for r in rows if r["encoder"] == prim and r.get("vs_best", {}).get("wilcoxon_p", 0) > 0.05]
        if close:
            rec = close[0]                            # pas de différence significative -> on garde l'encodeur principal (CLIP : a une tour texte)
    out = {"selected": best, "recommended": rec["config"],
           "recommendation_rule": "meilleur CIDEr sur val ; si l'encodeur principal n'est pas significativement moins bon (Wilcoxon p>0.05), on le garde", "criterion": "CIDEr moyen sur val (NN-caption), départage Jaccard classes@1",
           "n_val_images": int(len(qid)), "candidates": rows,
           "warning": "Avec 100 images, des écarts < ~0.05 CIDEr ne sont pas significatifs : voir wilcoxon_p. "
                      "Si p>0.05 contre le 2e, conserver l'encodeur le plus standard (CLIP ViT-B/32) et le documenter."}
    write_json(p["tables"].parent / "audit" / "preproc_config_selection.json", out)
    for r in rows:
        log.info("%-22s CIDEr=%.3f (cv10 %.3f±%.3f) IC95=[%.3f,%.3f] J@1=%.3f %s", r["config"], r["cider_mean"], r["cider_cv10_mean"], r["cider_cv10_std"],
                 r["ci95"][0], r["ci95"][1], r["jaccard_top1"], r.get("vs_best", ""))
    log.info("-> meilleur sur val : %s | recommandé : %s", best, rec["config"])

## Lancer l'étape

In [12]:
import sys
sys.argv = ["notebook"]          # pas d'arguments CLI dans un notebook
try:
    main()
except SystemExit as e:               # certains scripts terminent par sys.exit(code)
    print("terminé, code de sortie :", e.code)

20:24:08 | INFO | clip-b32|img2txt       CIDEr=0.558 (cv10 0.558±0.154) IC95=[0.463,0.659] J@1=0.535 
20:24:08 | INFO | dinov2-s|img2img       CIDEr=0.554 (cv10 0.554±0.081) IC95=[0.470,0.652] J@1=0.521 {'mean_diff': 0.0043062625066672515, 'wilcoxon_p': 0.7283878786577169}
20:24:08 | INFO | clip-b32|img2img       CIDEr=0.529 (cv10 0.529±0.213) IC95=[0.438,0.623] J@1=0.538 {'mean_diff': 0.02983945199039836, 'wilcoxon_p': 0.7140066154728504}
20:24:08 | INFO | debug-hist|img2img     CIDEr=0.086 (cv10 0.086±0.054) IC95=[0.043,0.135] J@1=0.130 {'mean_diff': 0.472683201650068, 'wilcoxon_p': 3.239224407217651e-14}
20:24:08 | INFO | debug-hist|img2txt     CIDEr=0.026 (cv10 0.026±0.018) IC95=[0.013,0.041] J@1=0.056 {'mean_diff': 0.5328802476171438, 'wilcoxon_p': 1.5701536176345187e-16}
20:24:08 | INFO | -> meilleur sur val : clip-b32|img2txt | recommandé : clip-b32|img2txt


## Résultats

In [13]:
import json, pandas as pd
from IPython.display import Image, Markdown, display
from common import load_config, paths
cfg = load_config(); P = paths(cfg)

### Résultats

In [14]:
r = json.load(open(P["audit"] / "preproc_config_selection.json"))
display(pd.DataFrame([{"config": c["config"], "CIDEr": round(c["cider_mean"], 3), "CV10 moy": round(c["cider_cv10_mean"], 3), "CV10 std": round(c["cider_cv10_std"], 3),
                       "IC95 bas": round(c["ci95"][0], 3), "IC95 haut": round(c["ci95"][1], 3), "Jaccard@1": round(c["jaccard_top1"], 3),
                       "p Wilcoxon vs meilleur": c.get("vs_best", {}).get("wilcoxon_p")} for c in r["candidates"]]))
print("MEILLEUR SUR VAL :", r["selected"]); print("RECOMMANDÉ (règle de significativité) :", r.get("recommended")); print(r["warning"])

,config,CIDEr,CV10 moy,CV10 std,IC95 bas,IC95 haut,Jaccard@1,p Wilcoxon vs meilleur
0,clip-b32|img2txt,0.558,0.558,0.154,0.463,0.659,0.535,NaN
1,dinov2-s|img2img,0.554,0.554,0.081,0.470,0.652,0.521,7.283879e-01
2,clip-b32|img2img,0.529,0.529,0.213,0.438,0.623,0.538,7.140066e-01
3,debug-hist|img2img,0.086,0.086,0.054,0.043,0.135,0.130,3.239224e-14
4,debug-hist|img2txt,0.026,0.026,0.018,0.013,0.041,0.056,1.570154e-16


MEILLEUR SUR VAL : clip-b32|img2txt
RECOMMANDÉ (règle de significativité) : clip-b32|img2txt
Avec 100 images, des Ã©carts < ~0.05 CIDEr ne sont pas significatifs : voir wilcoxon_p. Si p>0.05 contre le 2e, conserver l'encodeur le plus standard (CLIP ViT-B/32) et le documenter.


In [14]:
import json
import numpy as np
from pathlib import Path
from common import load_config, paths, sha256_file
from rag_data import Release

cfg = load_config(); P = paths(cfg)
rng = np.random.RandomState(cfg["seed"])
jp = Path("artifacts/audit/preproc_config_selection.json")
r = json.loads(jp.read_text(encoding="utf-8"))
r["candidates"] = [c for c in r["candidates"] if "debug-hist" not in c["config"]]   # <-- ligne ajoutée

def per_img(config):
    enc, mode = config.split("|")
    R = Release(P["art"], enc)
    ids, Q, refs = R.ids("val"), R.image_embeddings("val"), R.eval_refs("val")
    if mode == "img2img":
        _, I = R.search_images(Q, k=1, query_ids=ids)
        res = {int(i): R.kb_captions_of(I[k, 0])[0] for k, i in enumerate(ids)}
    else:
        _, cids, _ = R.search_texts(Q, k=1)
        capmap = {c["caption_id"]: c["caption_raw"] for c in R.kb_captions()}
        res = {int(i): capmap[int(cids[k, 0])] for k, i in enumerate(ids)}
    return per_image_cider(res, refs)

for c in r["candidates"]:
    x = per_img(c["config"])
    boots = np.array([rng.choice(x, len(x), replace=True).mean() for _ in range(2000)])
    c["cider_mean"] = float(x.mean())
    c["cider_boot_mean"] = float(boots.mean())
    c["cider_boot_std"] = float(boots.std())
    c["ci95"] = [float(np.percentile(boots, 2.5)), float(np.percentile(boots, 97.5))]
    c.pop("cider_cv10_mean", None); c.pop("cider_cv10_std", None)
    print(f'{c["config"]:20s} CIDEr={c["cider_mean"]:.3f}  bootstrap={c["cider_boot_mean"]:.3f}±{c["cider_boot_std"]:.3f}  IC95=[{c["ci95"][0]:.3f}, {c["ci95"][1]:.3f}]')

r["method_note"] = "IC95 par bootstrap (2000 rééchantillonnages) sur les 100 images val ; pas de validation croisée car rien n'est entraîné."
jp.write_text(json.dumps(r, indent=2, ensure_ascii=False), encoding="utf-8")

mp = Path("artifacts/manifest.json")
m = json.loads(mp.read_text(encoding="utf-8"))
info = m["files"]["audit/preproc_config_selection.json"]
info["sha256"] = sha256_file(jp)
for key in ("size", "bytes"):
    if key in info:
        info[key] = jp.stat().st_size
mp.write_text(json.dumps(m, indent=2, ensure_ascii=False), encoding="utf-8")
print("fichiers modifiés ou manquants :", Release("artifacts", "clip-b32").verify())

clip-b32|img2txt     CIDEr=0.558  bootstrap=0.557±0.049  IC95=[0.466, 0.657]
dinov2-s|img2img     CIDEr=0.554  bootstrap=0.554±0.048  IC95=[0.463, 0.652]
clip-b32|img2img     CIDEr=0.529  bootstrap=0.531±0.049  IC95=[0.441, 0.632]
fichiers modifiés ou manquants : []
